<a href="https://colab.research.google.com/github/AMIRMOHAMMAD-OSS/Phaseek/blob/main/phaseek.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phaseek — LLPS Protein Scorer (Colab Demo)

Phaseek is a lightweight tool for estimating liquid–liquid phase separation (LLPS) propensity from protein sequences.

**Use it two ways:**
1) paste a single amino acid sequence (we’ll save it to a temporary FASTA), or  
2) provide a FASTA filepath (for multiple sequences).  

**Artifacts** (plots/tables) are saved under `./Outputs/<Directory>/<ID>/`.

_Tip_: keep batched FASTA files to ~100 sequences for smoother performance.

In [ ]:
#@title Install dependencies (quiet)
%%capture
!pip install --upgrade pip
!pip install biopython plotly command_runner transformers ipympl stmol biotite requests
!apt-get update -qq && apt-get install -y -qq ncbi-blast+


In [ ]:
#@title Fetch Phaseek code
import os, shutil, json, textwrap, glob
from google.colab import output

REPO_URL = "https://github.com/AMIRMOHAMMAD-OSS/Phaseek"
REPO_DIR = "/content/Phaseek"

if not os.path.isdir(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull -q
%cd {REPO_DIR}/Functions
print("Phaseek ready ✅")

In [ ]:
#@title Check device
import torch
if torch.cuda.is_available():
    dev = torch.device("cuda")
    print("CUDA available:", torch.cuda.get_device_name())
else:
    dev = torch.device("cpu")
    print("CUDA not available — using CPU")

## Inputs
Paste a single sequence or supply a path to a FASTA file. If you paste a sequence, we’ll write it to `./tmp_input.fasta` for you.

In [ ]:
#@title Provide sequence or FASTA, plus run options
from command_runner import command_runner
import re, io, pathlib

#@markdown **One of these**: either paste a single sequence or give a FASTA path.
pasted_sequence = ""  #@param {type:"string"}
fasta_path = ""  #@param {type:"string"}

#@markdown **Run parameters**
ID = "DemoProtein"  #@param {type:"string"}
Directory = "Demo_Run"  #@param {type:"string"}
End_sequence = 512  #@param {type:"slider", min:0, max:5000, step:1}
plot_outputs = True  #@param {type:"boolean"}

def sanitize(seq: str) -> str:
    seq = seq.strip().upper()
    return re.sub(r"[^ACDEFGHIKLMNPQRSTVWY]", "", seq)

tmp_fasta = None
if pasted_sequence and not fasta_path:
    seq = sanitize(pasted_sequence)
    assert len(seq) > 0, "Pasted sequence became empty after sanitization."
    tmp_fasta = "/content/Phaseek/Functions/tmp_input.fasta"
    with open(tmp_fasta, "w") as f:
        f.write(">colab_input\n" + seq + "\n")
    fasta_path = tmp_fasta

if not fasta_path:
    raise ValueError("Please paste a sequence or supply a FASTA path.")

if not os.path.isfile(fasta_path):
    raise FileNotFoundError(f"FASTA file not found: {fasta_path}")

print("Using FASTA:", fasta_path)
print("ID:", ID)
print("Directory:", Directory)
print("End_sequence:", End_sequence)
print("Plot:", plot_outputs)

In [ ]:
#@title Run Phaseek
import shlex
cmd = (
    f"python runner.py --sequence {shlex.quote(fasta_path)} "
    f"--end_sequence {int(End_sequence)} --plot {str(bool(plot_outputs))} "
    f"--directory {shlex.quote(Directory)} --id {shlex.quote(ID)}"
)
print("Command:\n", cmd)
exit_code, live = command_runner(cmd, shell=True, live_output=True)
print("\nExit code:", exit_code)
if exit_code != 0:
    raise SystemExit("Phaseek run failed — check logs above.")

## Results & Artifacts

In [ ]:
#@title Browse generated outputs
from pathlib import Path
import IPython

base = Path(REPO_DIR) / "Functions" / "Outputs" / Directory / ID
if base.exists():
    print("Artifacts dir:", str(base))
    imgs = sorted([p for p in base.rglob("*.png")])
    csvs = sorted([p for p in base.rglob("*.csv")])
    pdfs = sorted([p for p in base.rglob("*.pdf")])
    print(f"Found {len(imgs)} image(s), {len(csvs)} csv(s), {len(pdfs)} pdf(s).\n")
    for p in imgs[:10]:
        display(IPython.display.Image(filename=str(p)))
    if csvs:
        print("First CSV preview:\n")
        import pandas as pd
        df = pd.read_csv(csvs[0])
        display(df.head(10))
else:
    print("No outputs found at:", str(base))

### Notes
- For single-sequence runs (pasted), the score may also print in the logs.
- Your outputs live in `Functions/Outputs/<Directory>/<ID>/` and travel with the runtime unless you move them to Drive.